In [34]:
import nibabel as nib
import numpy as numpy
import vtk
import tomllib
from vtk.util import numpy_support
import nrrd


case = ""
sequence = "T1"

with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/config.toml", mode="rb") as f:
    config = tomllib.load(f)


if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]


if case == "cut":
    input_file = input_file[:-4] + "_cropped_denoised.nii"



mesh_offset = numpy.loadtxt(base + output_folder + "mesh_offset_"+sequence+".txt")
output_file     = input_file[:-4] + ".vti"

In [35]:
segment_data, segment_header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in segment_header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(segment_header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)


original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


In [36]:
affine = numpy.eye(4)
affine[:3, :3] = dirs.T   # columns = axis direction vectors
affine[:3, 3] = origin

print(affine)
print("shape = ", segment_data.shape)

crop_affine = affine

[[ 3.12463187e-01  4.16039573e-03  3.81929878e-03 -5.59561996e+01]
 [-4.24688765e-03  3.12252430e-01  1.87024709e-02 -9.02006989e+01]
 [-2.22955103e-03 -1.17201089e-02  4.99635557e-01  2.04909000e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
shape =  (145, 209, 101)


In [37]:
def load_nifti(input_path, output_path, zoom_factor):

    nii = nib.load(input_path)
    data = numpy.array(nii.dataobj)  # efficient loading
    affine = nii.affine.copy()
    header = nii.header.copy()
    offset = affine[:3, 3] 

    pixel_spacing = header.get_zooms()[:3] 

    return data, pixel_spacing, offset, affine

data, pixel_spacing, offset, affine = load_nifti(input_file, output_file, 1)
print()
print("pixel_spacing = ", affine)
print("Original matrix shape = ", data.shape)

full_affine = affine


/var/folders/l0/d5k4byxx3690l9b12qr7mm440000gp/T/ipykernel_34963/3939849221.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  data = numpy.array(nii.dataobj)  # efficient loading



pixel_spacing =  [[-3.12463194e-01 -4.16039582e-03 -3.81932268e-03  1.13025139e+02]
 [ 4.24688915e-03 -3.12252402e-01 -1.87024865e-02  1.20845985e+02]
 [-2.22956575e-03 -1.17201172e-02  4.99635577e-01  4.36644363e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Original matrix shape =  (704, 704, 180)


In [38]:
A_full = numpy.array(full_affine)           # RAS
A_crop = numpy.array(crop_affine)           # LPS
A_crop_ras = numpy.diag([-1, -1, 1, 1]) @ A_crop   # convert crop to RAS

start = numpy.linalg.inv(A_full) @ A_crop_ras[:, 3]
start = numpy.round(start[:3]).astype(int)

print(start)

[181  98  35]
